# 5.1 Receivers

Receivers define what the solver records. A receiver group combines a device definition (the components measured by each receiver) with a coordinate set (where those receivers are placed). This tutorial builds a dense, multi-component elastic survey, runs it, and plots each recorded component so the component choices are tied directly to trace output.

The geometry uses one source and a receiver line. That keeps the tutorial focused on receiver semantics rather than survey scale: every receiver in the group records every component for the single source.

By the end, you should be able to design a receiver device, attach multiple components, place a dense receiver group, and connect trace components back to device definitions.


## How To Read This Tutorial

Receivers are devices, not just coordinate arrays. A device can carry multiple components, and a receiver group places that device at many coordinates. This separation is what lets large surveys remain concise while still recording pressure, velocity, displacement, strain, or other supported fields.

This notebook builds a dense survey so the result shape is easy to understand before the sparse and DAS tutorials introduce more specialized sampling.

## Supported Fields By Physics

The exact set of fields available in a run depends on the solver physics and output implementation, but the receiver component vocabulary follows this pattern:

| Physics | Common receiver fields | Direction required? | Notes |
| --- | --- | --- | --- |
| Acoustic | `pressure`, `velocity` | `velocity` needs a direction | Pressure is scalar; velocity components use physical directions such as `[1, 0]` or `[0, 1]`. |
| Elastic | `velocity`, `displacement`, `stress`, `strain`, `pressure` | Vector/tensor projections need a direction | Pressure is a derived scalar where available; strain and stress are projected along the supplied direction. |
| Poroelastic | `velocity`, `pressure`, `fluid_flux`, `fluid_displacement`, `stress`, `strain` | Vector/tensor projections need a direction | Fluid and solid fields are physics-specific; inspect `traces.summary` after a run to confirm written components. |
| Coupled | Union of fields supported by active domains | Depends on field | Unsupported fields in a domain are ignored or absent from output; keep receivers close to the domains they are meant to measure. |

A device can contain multiple components, and all components are written for each receiver coordinate in the group.


## Receiver Design Notes

Receiver authoring has three levels. A `ReceiverNode` describes the physical device and its components. A component names the field to sample, such as velocity, pressure, displacement, or strain, and may also specify a direction. A receiver group places copies of that device at coordinates and gives the group a name for trace output.

Use this separation when building production surveys: define the device once, place it many times, then use `TraceDataset` group/component metadata to choose what to plot or export.


## Imports

The plotting cells use Matplotlib directly for multi-panel layouts and FrequenSolve's trace plotting helper for each gather.


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import frequensolve as fs

u = fs.ureg


## Build A Small Elastic Model

The model is intentionally simple: one elastic layer with a free top and PML on the truncation boundaries. Receiver components are independent of mesh generation, but a complete simulation makes the resulting trace shapes inspectable.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="receiver_components",
    path="./scratch/tutorials/receivers",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="receiver_components",
    physics="elastic",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(
    name="elastic_halfspace",
    properties={"Vp": 2.5 * u.km / u.s, "Vs": 1.2 * u.km / u.s, "Rho": 2.2 * u.g / u.cm**3},
)
model.add_surface(name="bottom", depth=0.5 * u.km)
sim += model

sim += model.hex_mesh_generator([8, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=30.0)
sim.mesh.set_source_grading(d0=0.02, d1=0.08, factor=2.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

model.plot("vp", figsize=(7, 3), aspect="equal")


## Define A Multi-Component Receiver Device

`ReceiverNode` is a point receiver. The component names are user-facing labels that later appear in the trace store. `field` chooses the physical quantity; `direction` projects vector or tensor fields onto a scalar channel. The dense receiver group below writes `v_x`, `v_z`, and tangential strain `eps_xx` for every receiver coordinate.


In [ ]:
acq = fs.Acquisition()
acq.add_source_group(kind="vector", coords=[[0.5, 0.05]], direction=[0.0, 1.0])

node = fs.ReceiverNode(name="three_component_node")
node.add_component(name="v_x", field="velocity", direction=[1.0, 0.0])
node.add_component(name="v_z", field="velocity", direction=[0.0, 1.0])
node.add_component(name="eps_xx", field="strain", direction=[1.0, 0.0])

receiver_coords = [[x, 0.04] for x in np.linspace(0.1, 0.9, 81)]
acq.add_receiver_group(name="surface_dense", device=node, coords=receiver_coords)
sim += acq

{
    "source_groups": len(acq.source_groups),
    "receiver_groups": [group.name for group in acq.receiver_groups],
    "receiver_count": len(receiver_coords),
    "components": [component.name for component in node.components],
}


## Run The Dense Receiver Survey

The run cell is strict. If the local site or solver is not configured, the notebook should fail here with the solver/log path visible. After a successful run, `traces.summary` is the first object to inspect because it reports groups, components, sources, and frequency coverage.


In [ ]:
sim += fs.Discretization()
sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

site = fs.Site()
job = fs.TimeDomainJob(
    name="time_receivers",
    simulation=sim,
    f_min=0.0,
    f_max=30.0,
    T_max=0.9,
)
result = site.submit(job).wait()
traces = result.traces(upscale=4)
traces.summary


## Plot Each Receiver Component

All three panels come from the same source and receiver coordinates. Differences between panels are therefore measurement differences, not geometry differences. This is the easiest way to verify that receiver components are doing what you expect before scaling up a survey.


In [ ]:
wavelet = fs.RickerWavelet(f=12.0)
group = "surface_dense"
source = traces.sources(group)[0]
components = traces.components(group)
component_gathers = {
    component: traces.td(group, component, source, wavelet, upscale=4, T_max=0.9)
    for component in components
}

A = max(float(np.nanstd(np.real(gather.values))) for gather in component_gathers.values())
A = 2.0 * A if A > 0 else None
fig, axes = plt.subplots(1, len(component_gathers), figsize=(4.5 * len(component_gathers), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (component, gather) in zip(axes, component_gathers.items()):
    fs.plot_gather(gather, ax=ax, A=A, cmap="gray", title=component)
fig.tight_layout()


## Compare Center-Receiver Waveforms

Gather plots are good for moveout and spatial continuity. A single receiver overlay is better for comparing phase, polarity, and relative amplitude across components.


In [ ]:
center = len(receiver_coords) // 2
fig, ax = plt.subplots(figsize=(9, 4))
for component, gather in component_gathers.items():
    trace = gather.isel(receiver=center)
    ax.plot(trace["time"].values, np.real(trace.values), label=component)
ax.set_xlabel("Time")
ax.set_ylabel("Amplitude")
ax.set_title(f"Receiver {center + 1} component comparison")
ax.legend()
fig.tight_layout()


## Before Moving On

After a receiver run, inspect components and source ids from the trace file instead of assuming them from the authoring code. The trace store is the final authority on what was written.

The power-user habit is to design receiver devices explicitly, then place them with receiver groups. That makes multi-component surveys readable and avoids duplicating coordinate definitions.

## Result Review Checklist

Receiver tutorials should prove both metadata and data behavior. First inspect the supported-field table and receiver component definitions; then inspect the trace panels and overlays.

| Artifact | What to confirm |
| --- | --- |
| Receiver device | Multiple components can live on one device and share coordinates. |
| Dense group | Every receiver records every requested component for the selected source. |
| Gather panels | Component-specific moveout/amplitude differences are visible. |
| Center overlay | Components align in time where expected but preserve distinct amplitudes/signs. |

When adding a new receiver component, verify that the requested field is supported by the simulation physics before assuming a solver failure is numerical.
